# Qwen2.5-1.5B QLoRA Fine-tuning

Fine-tune Qwen2.5-1.5B for instruction following using QLoRA on a free T4 GPU.

**Runtime:** Change to T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Install dependencies
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Config

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # or "Qwen/Qwen3-0.6B"
DATASET_NAME = "tatsu-lab/alpaca"
OUTPUT_DIR = "./qwen2.5-1.5b-qlora-ft"
HUB_REPO = None  # Set to "your-username/model-name" to push to HF Hub

# Training
EPOCHS = 3
MAX_SAMPLES = 2000  # Use None for full dataset
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 512

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_NAME}")

## Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
model.print_trainable_parameters()

## Load & Format Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split="train")
if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

def format_alpaca(example):
    if example.get("input"):
        return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"}
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

dataset = dataset.map(format_alpaca)
print(f"Dataset: {len(dataset)} samples")
print(f"Example:\n{dataset[0]['text'][:500]}")

## Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## Save & Test

In [ ]:
# Save model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

In [ ]:
# Test inference
test_prompts = [
    "### Instruction:\nExplain quantum computing in simple terms.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function to reverse a string.\n\n### Response:\n",
    "### Instruction:\nWhat are the benefits of exercise?\n\n### Response:\n",
]

model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = text.split("### Response:\n")[-1].strip()
    print(f"Q: {prompt.split('### Instruction:\n')[1].split('### Response:')[0].strip()}")
    print(f"A: {response[:200]}\n")

## Push to Hugging Face (optional)

In [ ]:
if HUB_REPO:
    from huggingface_hub import login
    login()  # Will prompt for token
    
    model.push_to_hub(HUB_REPO)
    tokenizer.push_to_hub(HUB_REPO)
    print(f"Pushed to https://huggingface.co/{HUB_REPO}")
else:
    print("Set HUB_REPO to push to Hugging Face Hub")

## Merge LoRA (optional)

Merge LoRA weights into base model for standalone deployment.

In [ ]:
# Uncomment to merge LoRA into base model
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained(f"{OUTPUT_DIR}-merged")
# tokenizer.save_pretrained(f"{OUTPUT_DIR}-merged")
# print(f"Merged model saved to {OUTPUT_DIR}-merged")